# Week 10 · Topic 1 — Connecting and Creating Tables

**Box we are building: STORE, for real**

The whole database is one file. No server, nothing to install — `import sqlite3`
is already in Python.

> **Already know this?** Skip to the *Topic 1 Challenge*, then help a neighbour.

---
## 1. Connecting opens the file, or creates it

In [1]:
import sqlite3

conn = sqlite3.connect("network.db")
cursor = conn.cursor()
print(type(conn))

<class 'sqlite3.Connection'>


### Your turn
Connect to a new database file of your own name and print its type.

In [2]:
import sqlite3

conn = sqlite3.connect("zara.db")
cursor = conn.cursor()
print(type(conn))


<class 'sqlite3.Connection'>


---
## 2. `CREATE TABLE` — defining the shape

`IF NOT EXISTS` avoids an error if it is already there. `PRIMARY KEY` is a
unique, auto-numbered row identifier.

In [2]:
cursor.execute("""
    CREATE TABLE IF NOT EXISTS devices (
        id     INTEGER PRIMARY KEY,
        name   TEXT NOT NULL,
        ip     TEXT UNIQUE NOT NULL,
        active INTEGER DEFAULT 1,
        type   TEXT
    )
""")
conn.commit()

### Your turn
Create a table called `notes` with `id` (primary key) and `text` (not null).

In [3]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS notes (
    id   INTEGER PRIMARY KEY,
    text TEXT NOT NULL
)
""")
conn.commit()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('devices',), ('notes',)]


---
## 3. Five storage classes — that is genuinely all of them

`INTEGER`, `REAL`, `TEXT`, `BLOB`, `NULL`. No separate date type — store dates
as `TEXT`, ISO format (`YYYY-MM-DD`), so they still sort correctly.

In [ ]:
cursor.execute("""
    CREATE TABLE readings (
        id       INTEGER PRIMARY KEY,
        sensor   TEXT NOT NULL,
        value    REAL NOT NULL,
        recorded TEXT NOT NULL
    )
""")
conn.commit()

### Your turn
Design a `CREATE TABLE` statement for log entries: `id`, `timestamp` (TEXT), `level` (TEXT), `message` (TEXT).

In [8]:
cursor.execute("""
CREATE TABLE IF NOT EXISTS log_entries (
    id INTEGER PRIMARY KEY,
    timestamp TEXT,
    level TEXT,
    message TEXT
)
""")
conn.commit()

cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
print(cursor.fetchall())

[('devices',), ('notes',), ('log_entries',)]


---
## 4. SQLite will not stop you being wrong

It is flexible about types — a string can land in a `REAL` column with no
error at all. Your own code has to enforce what the schema will not.

In [6]:
cursor.execute("INSERT INTO readings (sensor, value, recorded) VALUES (?, ?, ?)",
               ("temp-01", "not-a-number", "2026-09-08"))
conn.commit()

row = cursor.execute("SELECT value FROM readings").fetchone()
print(row, type(row[0]))    # stored as-is - no error, no warning

OperationalError: no such table: readings

### Your turn
Write a small function that converts a value to `float` *before* inserting it, so a bad value fails loudly instead of being stored silently.

In [5]:
def insert_reading(sensor, raw_value, date):

    valid_value = float(raw_value) 
    
    cursor.execute(
        "INSERT INTO readings (sensor, value, recorded) VALUES (?, ?, ?)", 
        (sensor, valid_value, date)
    )
    conn.commit()
    print(f"Successfully inserted {valid_value}")

insert_reading("temp-01", "98.6", "2026-09-08")


OperationalError: no such table: readings

---
## Topic 1 Challenge

Create a table `servers` with `id`, `hostname` (unique, required), `used`
(real) and `total` (real). Confirm it was created by inserting one row and
selecting it back.

In [ ]:
# your code here

---
## Same idea, three fields

| Field | Looks like |
|---|---|
| **AI / Data Science** | a `datasets` table, one row per dataset you have loaded |
| **Cyber Security** | an `incidents` table, one row per event you have logged |
| **IT** | a `devices` table, one row per server or service you track |

---
## Common mistakes

| Mistake | What you see | Fix |
|---|---|---|
| forgetting `conn.commit()` | changes silently vanish | commit after every write, or use `with` (Topic 3) |
| assuming a type is enforced | a string lands in a `REAL` column with no error | validate/convert in Python before inserting |
| querying a table that does not exist yet | `sqlite3.OperationalError: no such table` | run `CREATE TABLE` first, check the name matches exactly |
| forgetting `NOT NULL`/`UNIQUE` where it matters | duplicate or missing data slips in silently | decide your constraints when you design the schema, not after |

## Go deeper

`examples/01_connect_and_create.py` — connecting, cursors, CREATE TABLE, in-memory databases
`examples/02_data_types_and_schema.py` — the five storage classes, and SQLite's loose typing
Docs: [sqlite3 — Python docs](https://docs.python.org/3/library/sqlite3.html)

---
## Recap

| You can now | Using |
|---|---|
| connect to a database | `sqlite3.connect(path)` |
| create a table | `CREATE TABLE IF NOT EXISTS` |
| choose a type | `INTEGER`, `REAL`, `TEXT`, `BLOB`, `NULL` |
| save your changes | `conn.commit()` |

**Next:** Topic 2 — putting real data in, and getting it back out.